# CNN Architecture Notes

## LeNet

**Architecture pattern**

```text
Conv2d -> Sigmoid -> AvgPool2d

-> Conv2d -> Sigmoid -> AvgPool2d
-> Flatten
-> Linear -> Sigmoid -> Linear -> Sigmoid -> Linear
-> logits
```

**Purpose / idea**

- LeNet is a small, auditable CNN that turns image grids into class scores.
- Convolution learns local spatial features with shared weights, pooling
summarizes nearby spatial regions, flattening converts feature maps into a vector, and dense layers produce class logits.
- Sigmoid is used as an intermediate activation. It squashes activations toward (0, 1), but the final output is still logits: raw, unnormalized class
scores.

## AlexNet

**Architecture pattern**

```text
Large early Conv2d -> ReLU -> MaxPool2d

-> deeper Conv2d/ReLU stack
-> MaxPool2d
-> Flatten
-> Linear -> ReLU -> Dropout
-> Linear -> ReLU -> Dropout
-> Linear
-> logits
```

**Purpose / idea**

- AlexNet scales the LeNet idea into a deeper CNN.
- It learns visual representations directly from data instead of relying on hand-engineered features.
- **It reduces spatial resolution while increasing channel capacity, so later layers represent richer features over smaller feature maps**.
- ReLU replaces sigmoid-style activations because it is easier to optimize in deeper networks.
- Dropout regularizes the dense classifier head.

## VGG

**Architecture pattern**

```text
Repeated VGG block:

Conv2d(3x3, padding=1) -> ReLU
[repeat one or more times]
-> MaxPool2d

Final head in the tutorial:

AdaptiveAvgPool2d -> Flatten -> Linear
```

**Purpose / idea**

- VGG’s main idea is architectural regularity: build deep CNNs by repeating simple blocks of small 3x3 convolutions followed by pooling.
- **Stacked 3x3 convolutions increase the effective receptive field, add extra nonlinearities, and keep the design easy to scale**.
- VGG reduces spatial size with `MaxPool2d` while increasing channels across stages.
- `AdaptiveAvgPool2d` is used in the tutorial’s final head, but it is not the defining replacement for max pooling inside VGG blocks.

## NiN

**Architecture pattern**

```text
NiN block:

Spatial Conv2d -> ReLU
-> 1x1 Conv2d -> ReLU
-> 1x1 Conv2d -> ReLU

Classifier:

Conv2d(final_channels = num_classes, kernel_size=1)
-> AdaptiveAvgPool2d
-> Flatten
-> logits
```

**Purpose / idea**

- NiN, or Network in Network, adds small per-location networks inside the CNN.
- The 1x1 convolutions mix channels independently at each spatial location,
like shared linear layers applied across all pixels.
- NiN replaces a large dense classifier head with global average pooling.
- The final class channels are averaged over spatial positions to become logits.
- **This greatly reduces classifier parameters but discards exact final-map location information**.

## GoogLeNet

**Architecture pattern**

```text
Stem Conv2d/ReLU

-> Inception block
-> pooling
-> Inception block
-> AdaptiveAvgPool2d
-> Flatten
-> Linear
-> logits

Inception block pattern

branch 1: 1x1 Conv
branch 2: 1x1 Conv -> 3x3 Conv
branch 3: 1x1 Conv -> 5x5 Conv
branch 4: MaxPool2d -> 1x1 Conv

Concatenate branch outputs along channel dimension
```

**Purpose / idea**

- GoogLeNet’s defining idea is the Inception block. **Several branches process the same input at different receptive-field scales, then their outputs are concatenated along the channel dimension**.
- The 1x1 convolutions often act as bottlenecks: they reduce channel width before expensive larger convolutions, lowering parameter count and compute.

## ResNet

**Architecture pattern**

```text
Stem Conv2d -> BatchNorm -> ReLU

-> ResidualBlock(s)
-> AdaptiveAvgPool2d
-> Flatten
-> Linear
-> logits

Residual block idea

output = learned_update(X) + shortcut(X)
```

**Purpose / idea**

- ResNet’s main contribution is residual learning. **Instead of forcing a block to learn a full new transformation from scratch, the block learns a correction
to the existing representation**.

- If the best transformation is close to identity, the residual branch can learn something near zero and the shortcut can preserve the input. This makes
very deep networks easier to optimize.

- BatchNorm helps stabilize channel activation scale, but it is not the defining replacement for pooling. The defining mechanism is the shortcut/residual
addition.

## DenseNet

**Architecture pattern**

```text
Stem Conv2d

-> DenseBlock
-> TransitionBlock
-> DenseBlock
-> AdaptiveAvgPool2d
-> Flatten
-> Linear
-> logits

Dense block idea

new representation = concatenate(old features, newly computed features)

Transition block

BatchNorm -> ReLU -> 1x1 Conv2d -> AvgPool2d
```

**Purpose / idea**

- DenseNet preserves earlier features by concatenating them with newly computed features. Every later layer in a dense block receives all earlier feature
maps as input.

- This causes channel count to grow by the growth rate. **Transition blocks control that growth by compressing channels with 1x1 convolutions and reducing
spatial size with average pooling**.

- **ResNet combines old and new information by addition. DenseNet combines them by concatenation**.